In [ ]:
import pickle

import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt

from tools.geometry import generate_detector
from tools.generate import read_photon_data_from_photonsim
from tools.optimization.optimize import load_optimization_config, get_detector_params_from_config

print("Imports successful")

In [ ]:
import pickle
import numpy as np

# --------------------------
# Load results
# --------------------------
results_file = '../output/single_ring_optimization_adam_20251009_124113_events.pkl'

print(results_file)

with open(results_file, 'rb') as f:
    results_summary = pickle.load(f)

print(f"Loaded results from: {results_file}")
print(f"\nConfiguration:")
print(f"  Optimizer: {results_summary['config'].get('optimizer', 'SGD')}")
print(f"  Number of events: {results_summary['config']['n_events']}")
print(f"  Temperature: {results_summary['config']['temperature']}")

# --------------------------
# Extract event results
# --------------------------
all_event_results = results_summary['all_event_results']
n_events = len(all_event_results)

print(f"\nSuccessfully loaded {n_events} events")

# --------------------------
# Extract convergence histories
# --------------------------
def extract_histories(all_event_results):
    """
    Extract convergence histories for all events.
    
    Returns:
        Dictionary with arrays for each metric, shape (n_events, n_iterations)
    """
    histories = {
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
        'combined_losses': [],
        'vertex_losses': [],
        'counts_losses': [],
        'energy_losses': []
    }
    
    for event_result in all_event_results:
        opt_results = event_result['optimization_results']
        history = opt_results['history']
        
        histories['position_errors'].append(history['position_errors'])
        histories['direction_errors'].append(history['direction_errors'])
        histories['t0_errors'].append(history['t0_errors'])
        histories['energy_errors'].append(history['energy_errors'])
        histories['combined_losses'].append(history['combined_losses'])
        histories['vertex_losses'].append(history['vertex_losses'])
        histories['counts_losses'].append(history['counts_losses'])
        histories['energy_losses'].append(history['energy_losses'])
    
    # Convert to numpy arrays
    for key in histories:
        histories[key] = np.array(histories[key])
    
    return histories

histories = extract_histories(all_event_results)

# --------------------------
# Compute statistics
# --------------------------
def compute_statistics(data_array):
    """
    Compute mean, median, 68th percentile, and 90th percentile.
    
    Args:
        data_array: numpy array of shape (n_events, n_iterations)
    
    Returns:
        Dictionary with statistical measures
    """
    return {
        'mean': np.mean(data_array, axis=0),
        'median': np.median(data_array, axis=0),
        'percentile_68': np.percentile(data_array, 68, axis=0),
        'percentile_90': np.percentile(data_array, 90, axis=0)
    }

# Compute statistics for all metrics
stats = {}
for key in histories:
    stats[key] = compute_statistics(histories[key])

print("Computed statistics for all metrics")

n_events, n_iterations = histories['position_errors'].shape
print(f"Extracted histories: {n_events} events, {n_iterations} iterations each")

# --------------------------
# Convert energy errors → momentum errors (%)
# --------------------------
# Constants
m_mu = 0.105658  # GeV (muon mass)
T_mu = 1050.0    # GeV (kinetic energy)
E_total = T_mu + m_mu
p_mu = np.sqrt(E_total**2 - m_mu**2)
conversion_factor = E_total / p_mu

print(f"\n--- Energy → Momentum conversion ---")
print(f"Muon mass = {m_mu:.6f} GeV")
print(f"Kinetic energy = {T_mu:.3f} GeV")
print(f"Total energy = {E_total:.6f} GeV")
print(f"Momentum = {p_mu:.6f} GeV")
print(f"Conversion factor (Δp/p)/(ΔE/E) = {conversion_factor:.8f}")

# If energy_errors are ABSOLUTE [GeV]:
momentum_errors_percent = (conversion_factor * (histories['energy_errors'] / E_total)) * 100

# If instead they are already RELATIVE (ΔE/E), comment the line above and use:
# momentum_errors_percent = conversion_factor * histories['energy_errors'] * 100

# Store and compute statistics
histories['momentum_errors_percent'] = momentum_errors_percent
stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)

print("\nConverted energy errors into momentum errors (in %) and computed statistics.")
print(f"Example: mean momentum error at last iteration = {stats['momentum_errors_percent']['mean'][-1]:.4f}%")


In [ ]:
import pandas as pd
import numpy as np

# Number of events and iterations
n_iterations, n_events = histories['position_errors'].shape

# --- Last iteration 68% errors ---
distance_68_last = np.percentile(histories['position_errors'][:, -1], 68)*100
angle_68_last    = np.percentile(histories['direction_errors'][:, -1], 68)
t0_68_last   = np.percentile(histories['t0_errors'][:, -1], 68)
momentum_68_last   = np.percentile(histories['momentum_errors_percent'][:, -1], 68)

# # --- Build table ---
data = {
    "Distance 68% error (cm)": [distance_68_last],
    "Angle 68% error (°)":    [angle_68_last],
    "t0 error (ns)": [t0_68_last],
    "Momentum 68% error (%)": [momentum_68_last]
}

data = {
    "Distance error (cm)": [distance_68_last],
    "Angle error (°)":    [angle_68_last],
    "t0 error (ns)": [t0_68_last],
    "Momentum 68% error (%)": [momentum_68_last]
}
index = ["LUCiD"]

df = pd.DataFrame(data, index=index)
print("\n" + "="*80)
print("68% METRICS")
print("="*80)
print(df.to_string(float_format=lambda x: f"{x:.2f}"))
print("="*80 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

def create_convergence_plot_v2(
    histories, stats, metric_name, ylabel, title,
    use_log=False, figsize=(4, 2.5),
    show_histogram=True, hist_bins=30, hist_width=0.2, max_y=1.0
):
    fig, ax = plt.subplots(figsize=figsize)
    n_iterations = len(stats[metric_name]['mean'])
    n_events = len(histories[metric_name])
    iterations = np.arange(n_iterations)

    col_mean, col_median, col_p68, col_p90 = np.array([
        'red',
        'darkorange', 
        'cornflowerblue',
        'springgreen',
    ])


    
    lw = 1.5

    # Plot main curves
    ax.plot(iterations, stats[metric_name]['mean'],
            color=col_mean, linewidth=lw, label=f'Mean', zorder=3)
    ax.plot(iterations, stats[metric_name]['median'],
            color=col_median, linewidth=lw, linestyle='--', label='Median', zorder=2)
    ax.plot(iterations, stats[metric_name]['percentile_68'],
            color=col_p68, linewidth=lw, linestyle='-.', label='68%', zorder=1)
    ax.plot(iterations, stats[metric_name]['percentile_90'],
            color=col_p90, linewidth=lw, linestyle=':', label='90%', zorder=1)

    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, n_iterations - 1)
    ax.set_ylim(0, max_y)
    
    # Format y-axis to always show consistent number of decimal places
    # Use consistent formatting to ensure histogram alignment across all plots
    # Format: 3 digits before decimal, 2 after (e.g., "100.00", "  5.00", "  1.50")
    ax.yaxis.set_major_formatter(FormatStrFormatter('%6.2f'))
    
    ax.legend(loc='best', framealpha=0., frameon=False, ncol=2, handlelength=2.0, columnspacing=1.0)

    if use_log:
        ax.set_yscale('log')

    # --- Optional histogram on the right ---
    if show_histogram:
        # Create a small inset axis on the right
        bbox = ax.get_position()
        hist_ax = fig.add_axes([
            bbox.x1 + 0.003,  # X position (a bit right of main plot)
            bbox.y0,         # Same Y position
            0.2,#hist_width * bbox.width,  # Width relative to main axis
            bbox.height       # Same height
        ])

        # Extract values at last iteration
        last_iter_values = histories[metric_name][:, -1]

        # Plot histogram horizontally
        hist_ax.hist(last_iter_values, bins=hist_bins, orientation='horizontal',
                     color='gray', alpha=0.2, edgecolor='black', linewidth=0.5)

        # Add horizontal lines for statistics at last iteration
        last_mean = stats[metric_name]['mean'][-1]
        last_median = stats[metric_name]['median'][-1]
        last_p68 = stats[metric_name]['percentile_68'][-1]
        last_p90 = stats[metric_name]['percentile_90'][-1]
        
        # Plot horizontal lines with same colors and styles as main plot
        hist_ax.axhline(y=last_mean, color=col_mean, linewidth=lw, linestyle='-', alpha=0.7)
        hist_ax.axhline(y=last_median, color=col_median, linewidth=lw, linestyle='--', alpha=0.7)
        hist_ax.axhline(y=last_p68, color=col_p68, linewidth=lw, linestyle='-.', alpha=0.7)
        hist_ax.axhline(y=last_p90, color=col_p90, linewidth=lw, linestyle=':', alpha=0.7)

        hist_ax.set_yticks([])  # No y ticks
        hist_ax.set_xticks([])
        #hist_ax.set_title('Final dist.', fontsize=8)
        hist_ax.set_ylim(ax.get_ylim())  # Match vertical scale
        sns.despine(ax=hist_ax, left=True, bottom=False, right=False, top=False)
        #sns.despine(ax=hist_ax, left=True, bottom=True, right=True, top=True)
        ax.grid(True, alpha=0.3)
    #plt.tight_layout()
    return fig, ax

fig, ax = create_convergence_plot_v2(
    histories, stats, 'position_errors',
    ylabel='Position Error (m)',
    title='Position Error Convergence',
    show_histogram=True, max_y=1.0, hist_bins=15
)
plt.savefig('figures/convergence_position_error_with_hist.png', dpi=150, bbox_inches='tight')
plt.show()


fig, ax = create_convergence_plot_v2(
    histories, stats, 'direction_errors',
    ylabel='Direction Error (degrees)',
    title='Direction Error Convergence',
    show_histogram=True, max_y=5.0
)
plt.savefig('figures/convergence_direction_error.png', dpi=150, bbox_inches='tight')
plt.show()

# t0 error convergence
fig, ax = create_convergence_plot_v2(
    histories, stats, 't0_errors',
    ylabel='t0 Error (ns)',
    title='t0 Error Convergence',
    show_histogram=True, max_y=3.0
)
plt.savefig('figures/convergence_t0_error.png', dpi=150, bbox_inches='tight')
plt.show()

# Momentum error convergence
fig, ax = create_convergence_plot_v2(
    histories, stats, 'momentum_errors_percent',
    ylabel='Momentum Error (%)',
    title='Momentum Error Convergence',
    show_histogram=True, max_y=5.0
)
plt.savefig('figures/convergence_momentum_error.png', dpi=150, bbox_inches='tight')
plt.show()